# RetailMind AI - Day 3: Multi-Model Comparison, Tuning & Bias-Variance

**Goal:** tune and validate competing churn models, diagnose whether each one is underfitting or overfitting, and lock a robust champion as `models/churn_v3.joblib`.

**This increment uses only this day's scikit-learn tooling:** `GridSearchCV`, cross-validation (`StratifiedKFold`, `cross_validate`) and `learning_curve`, plus pandas/numpy for data handling and joblib for persistence.

**What this notebook does**

1. Rebuilds the Day 1/Day 2 preprocessing and stratified split.
2. Defines a hyperparameter grid per candidate model.
3. Runs `GridSearchCV` with 5-fold cross-validation and multiple metrics.
4. Compares **train vs validation** scores to read off bias (underfit) vs variance (overfit).
5. Uses **learning curves** to decide whether more data, more features or more regularisation is needed.
6. Locks the tuned champion, evaluates it once on the held-out test set, and writes a model card with the chosen hyperparameters and CV scores (ready to paste into the README).

**Artifacts written**

| File | Contents |
| --- | --- |
| `models/churn_v3.joblib` | Tuned champion pipeline |
| `models/day3_cv_results.csv` | CV leaderboard for every candidate |
| `models/day3_test_metrics.csv` | One-shot holdout metrics for the champion |
| `models/day3_best_params.json` | Champion + per-model best hyperparameters |
| `models/day3_model_card.md` | README-ready documentation block |

## 1. Import libraries

`GridSearchCV` handles the search, `StratifiedKFold` keeps the churn ratio stable in every fold, `cross_validate` reports several metrics per fold, and `learning_curve` measures the bias-variance trade-off as training data grows.

`warnings` is filtered only to keep the notebook readable; warnings are expected for some metric/fold combinations (e.g. precision with no positive predictions).

In [1]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    GridSearchCV,
    learning_curve,
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
)

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

RANDOM_STATE = 42

## 2. Load the Telco Customer Churn dataset

The path check supports running the notebook from either `notebooks/` or the project root, matching the Day 2 notebook.

In [2]:
DATA_CANDIDATES = [
    Path("../data/Telco-Customer-Churn.csv"),
    Path("data/Telco-Customer-Churn.csv"),
]
DATA_PATH = next((path for path in DATA_CANDIDATES if path.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        "Dataset not found. Keep Telco-Customer-Churn.csv inside the data/ folder."
    )

df_raw = pd.read_csv(DATA_PATH)

print("Dataset:", DATA_PATH.resolve())
print("Rows:", df_raw.shape[0], "| Columns:", df_raw.shape[1])
df_raw.head()

Dataset: C:\Users\ANSHU SHAW\OneDrive\Desktop\RETAILMIND_AI\data\Telco-Customer-Churn.csv
Rows: 7043 | Columns: 21


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 3. Clean the data and make a stratified split

- `TotalCharges` contains blank strings, so it is converted to numeric and left as `NaN` for the pipeline imputer.
- `Churn` is mapped to `1` / `0`.
- `customerID` is dropped because it is an identifier, not customer behaviour.
- The 20% test set is created **once**, stratified, and never used for tuning. It is touched only at the end, so the final score is an honest estimate of generalisation.

In [3]:
rows_before = len(df_raw)

df = df_raw.copy()
df.columns = df.columns.str.strip()
df = df.drop_duplicates()
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})
df = df.dropna(subset=["Churn"]).reset_index(drop=True)

X = df.drop(columns=["Churn", "customerID"], errors="ignore")
y = df["Churn"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print("Rows removed as duplicates/null target:", rows_before - len(df))
print("Clean rows:", len(df))
print("Training rows:", len(X_train), "| Test rows:", len(X_test))
print("Churn rate - full: {:.3f} | train: {:.3f} | test: {:.3f}".format(
    y.mean(), y_train.mean(), y_test.mean()
))

Rows removed as duplicates/null target: 0
Clean rows: 7043
Training rows: 5634 | Test rows: 1409
Churn rate - full: 0.265 | train: 0.265 | test: 0.265


## 4. Preprocessing

Two column transformers are prepared because the candidates have different needs:

- **Trees** (Decision Tree, Random Forest) do not need scaling.
- **SVM and Logistic Regression** are scale-sensitive, so numeric columns are standardised.

Both impute missing numerics with the median and encode categoricals with one-hot. Imputers/encoders live **inside** each pipeline, so every CV fold learns them from its own training split. This prevents leakage during tuning.

In [4]:
numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=["number"]).columns.tolist()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

scaled_preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features),
])

tree_preprocessor = ColumnTransformer(transformers=[
    ("num", SimpleImputer(strategy="median"), numeric_features),
    ("cat", categorical_pipeline, categorical_features),
])

Numeric features: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
Categorical features: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


## 5. Candidate models and search grids

Four candidates from Day 1 and Day 2 are tuned. `class_weight="balanced"` is kept everywhere because churn is the minority class and missed churners are expensive.

The grids are deliberately compact: big enough to test the important bias-variance knobs, small enough to finish quickly.

| Model | Hyperparameters searched | What the grid controls |
| --- | --- | --- |
| Logistic Regression | `C` | Regularisation strength (smaller `C` = stronger regularisation) |
| Decision Tree | `max_depth`, `min_samples_leaf` | Tree depth and minimum leaf size; the main overfitting controls |
| Random Forest | `n_estimators`, `max_depth`, `min_samples_leaf` | Ensemble size and per-tree complexity |
| SVM (RBF) | `C`, `gamma` | Margin softness and kernel reach; the classic SVM bias-variance pair |

All keys are prefixed with `classifier__` because the estimator is the final step of a `Pipeline`.

Two practical notes:

- **SVM is searched with `probability=False`.** ROC-AUC scoring uses `decision_function`, so probabilities are not needed during tuning. If the SVM wins, section 10 calibrates it once on the training split before saving, because downstream scoring needs `predict_proba`.
- **SVM folds run sequentially (`n_jobs=1`).** Parallel libsvm workers can crash with an access violation on Windows; the tree/linear models still run in parallel. Sequential SVM tuning takes roughly 3 minutes.

In [5]:
candidates = {
    "Logistic Regression": (
        scaled_preprocessor,
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        ),
    ),
    "Decision Tree": (
        tree_preprocessor,
        DecisionTreeClassifier(
            class_weight="balanced",
            random_state=RANDOM_STATE,
        ),
    ),
    "Random Forest": (
        tree_preprocessor,
        RandomForestClassifier(
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=1,  # outer GridSearchCV already parallelises across folds
        ),
    ),
    "SVM (RBF)": (
        scaled_preprocessor,
        SVC(
            kernel="rbf",
            class_weight="balanced",
            random_state=RANDOM_STATE,
            # Calibrated in section 10 only if the SVM wins.
        ),
    ),
}

param_grids = {
    "Logistic Regression": {
        "classifier__C": [0.01, 0.1, 1.0, 10.0],
    },
    "Decision Tree": {
        "classifier__max_depth": [4, 6, 8, None],
        "classifier__min_samples_leaf": [1, 5, 20],
    },
    "Random Forest": {
        "classifier__n_estimators": [200],
        "classifier__max_depth": [None, 10, 20],
        "classifier__min_samples_leaf": [1, 5],
    },
    "SVM (RBF)": {
        "classifier__C": [0.1, 1.0, 10.0],
        "classifier__gamma": ["scale", 0.01],
    },
}

grid_overview = pd.DataFrame([
    {
        "Model": name,
        "Hyperparameters searched": ", ".join(
            f"{key.replace('classifier__', '')}={values}"
            for key, values in grid.items()
        ),
        "Combinations": int(np.prod([len(values) for values in grid.values()])),
    }
    for name, grid in param_grids.items()
])

grid_overview

,Model,Hyperparameters searched,Combinations
0,Logistic Regression,"C=[0.01, 0.1, 1.0, 10.0]",4
1,Decision Tree,"max_depth=[4, 6, 8, None], min_samples_leaf=[1...",12
2,Random Forest,"n_estimators=[200], max_depth=[None, 10, 20], ...",6
3,SVM (RBF),"C=[0.1, 1.0, 10.0], gamma=['scale', 0.01]",6


## 6. Tune every candidate with `GridSearchCV`

Design choices:

- **5-fold `StratifiedKFold`** with shuffling: each fold keeps the ~26% churn rate, and every configuration is scored five times.
- **Multi-metric scoring**: accuracy, precision, recall, F1 and ROC-AUC are all recorded, so the leaderboard can be read from different business angles.
- **`refit="roc_auc"`**: the champion is selected on ROC-AUC because it is threshold-independent and does not reward a model that simply predicts "no churn" for everyone. Recall is reported alongside and used as a tie-breaker when reading the leaderboard.
- **`return_train_score=True`**: needed to compare training-fold vs validation-fold performance, which is the bias-variance signal.
- **`n_jobs=-1`**: parallelises folds for the tree/linear models. SVC overrides this to `n_jobs=1` for Windows stability (see section 5). Total tuning runtime is roughly 4-6 minutes on a laptop.

In [6]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
}

searches = {}

# libsvm workers are unstable when several folds run in parallel on Windows.
search_jobs = {"SVM (RBF)": 1}

for model_name, (preprocessor, estimator) in candidates.items():
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", estimator),
    ])

    search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grids[model_name],
        scoring=scoring,
        refit="roc_auc",
        cv=cv,
        n_jobs=search_jobs.get(model_name, -1),
        return_train_score=True,
        verbose=0,
    )
    search.fit(X_train, y_train)
    searches[model_name] = search

    print(f"{model_name}: {len(search.cv_results_['params'])} configurations x 5 folds done")
    print("  best:", search.best_params_)
    print(f"  CV ROC-AUC: {search.best_score_:.4f}")

Logistic Regression: 4 configurations x 5 folds done
  best: {'classifier__C': 10.0}
  CV ROC-AUC: 0.8462


Decision Tree: 12 configurations x 5 folds done
  best: {'classifier__max_depth': 4, 'classifier__min_samples_leaf': 1}
  CV ROC-AUC: 0.8279


Random Forest: 6 configurations x 5 folds done
  best: {'classifier__max_depth': 10, 'classifier__min_samples_leaf': 5, 'classifier__n_estimators': 200}
  CV ROC-AUC: 0.8464


SVM (RBF): 6 configurations x 5 folds done
  best: {'classifier__C': 1.0, 'classifier__gamma': 0.01}
  CV ROC-AUC: 0.8424


## 7. Cross-validation leaderboard

Each row shows the best configuration for that model, its mean CV ROC-AUC with fold-to-fold standard deviation, plus CV recall/F1 and the mean **training-fold** ROC-AUC. The train-CV gap is computed here and interpreted in the next section.

In [7]:
rows = []

for model_name, search in searches.items():
    best_index = search.best_index_
    results = search.cv_results_

    rows.append({
        "Model": model_name,
        "Best Parameters": search.best_params_,
        "CV ROC-AUC": results["mean_test_roc_auc"][best_index],
        "CV ROC-AUC Std": results["std_test_roc_auc"][best_index],
        "CV Accuracy": results["mean_test_accuracy"][best_index],
        "CV Precision": results["mean_test_precision"][best_index],
        "CV Recall": results["mean_test_recall"][best_index],
        "CV F1": results["mean_test_f1"][best_index],
        "Train ROC-AUC": results["mean_train_roc_auc"][best_index],
    })

cv_summary = (
    pd.DataFrame(rows)
    .sort_values(["CV ROC-AUC", "CV Recall"], ascending=False)
    .reset_index(drop=True)
)

cv_summary["Train-CV Gap"] = cv_summary["Train ROC-AUC"] - cv_summary["CV ROC-AUC"]

cv_summary.round(4)

,Model,Best Parameters,CV ROC-AUC,CV ROC-AUC Std,CV Accuracy,CV Precision,CV Recall,CV F1,Train ROC-AUC,Train-CV Gap
0,Random Forest,"{'classifier__max_depth': 10, 'classifier__min...",0.8464,0.0093,0.7600,0.5332,0.7746,0.6314,0.9202,0.0738
1,Logistic Regression,{'classifier__C': 10.0},0.8462,0.0127,0.7481,0.5164,0.8000,0.6275,0.8498,0.0036
2,SVM (RBF),"{'classifier__C': 1.0, 'classifier__gamma': 0.01}",0.8424,0.0101,0.7128,0.4766,0.8254,0.6040,0.8491,0.0068
3,Decision Tree,"{'classifier__max_depth': 4, 'classifier__min_...",0.8279,0.0106,0.7233,0.4895,0.7920,0.6030,0.8418,0.0139


## 8. Diagnose bias vs variance

Read each model with three numbers:

- **CV score** — how well it generalises to unseen folds.
- **Train score** — how well it fits data it has seen.
- **Gap = train - CV** — the variance signal.

Rules used below (documented, deliberately simple):

| Pattern | Reading | Typical fix |
| --- | --- | --- |
| Gap > 0.05 | **High variance / overfitting** — memorises training folds | More regularisation, shallower trees, more data |
| CV < 0.70 and gap <= 0.05 | **High bias / underfitting** — too simple for the signal | More features, richer model, weaker regularisation |
| Gap <= 0.05 and CV >= 0.70 | **Balanced** — good generalisation | Keep, or squeeze with feature engineering |
| Large CV std | **Unstable** — sensitive to the particular split | More folds/data, stronger regularisation |

ROC-AUC of 0.70 is used as the "useful signal" line for this dataset; it is a reading aid, not a universal threshold.

In [8]:
def bias_variance_label(train_score, cv_score, gap_threshold=0.05, low_score=0.70):
    gap = train_score - cv_score
    if gap > gap_threshold:
        return "High variance (overfitting)"
    if cv_score < low_score:
        return "High bias (underfitting)"
    return "Balanced"


cv_summary["Diagnosis"] = [
    bias_variance_label(row["Train ROC-AUC"], row["CV ROC-AUC"])
    for _, row in cv_summary.iterrows()
]

for _, row in cv_summary.iterrows():
    print(f"{row['Model']}")
    print(f"  train={row['Train ROC-AUC']:.4f}  cv={row['CV ROC-AUC']:.4f}  "
          f"gap={row['Train-CV Gap']:+.4f}  std={row['CV ROC-AUC Std']:.4f}")
    print(f"  -> {row['Diagnosis']}\n")

cv_summary[[
    "Model", "Train ROC-AUC", "CV ROC-AUC", "Train-CV Gap",
    "CV ROC-AUC Std", "CV Recall", "Diagnosis",
]].round(4)

Random Forest
  train=0.9202  cv=0.8464  gap=+0.0738  std=0.0093
  -> High variance (overfitting)

Logistic Regression
  train=0.8498  cv=0.8462  gap=+0.0036  std=0.0127
  -> Balanced

SVM (RBF)
  train=0.8491  cv=0.8424  gap=+0.0068  std=0.0101
  -> Balanced

Decision Tree
  train=0.8418  cv=0.8279  gap=+0.0139  std=0.0106
  -> Balanced



,Model,Train ROC-AUC,CV ROC-AUC,Train-CV Gap,CV ROC-AUC Std,CV Recall,Diagnosis
0,Random Forest,0.9202,0.8464,0.0738,0.0093,0.7746,High variance (overfitting)
1,Logistic Regression,0.8498,0.8462,0.0036,0.0127,0.8000,Balanced
2,SVM (RBF),0.8491,0.8424,0.0068,0.0101,0.8254,Balanced
3,Decision Tree,0.8418,0.8279,0.0139,0.0106,0.7920,Balanced


## 9. Learning curves

The gap table says *whether* a model is biased or variance-heavy; the learning curve says *what to do next*. The two strongest candidates are plotted as tables (matplotlib is intentionally not used in this increment) at 10% to 100% of the training data, 5-fold CV:

- **Train and validation curves converge at a high score** -> the model is well specified; more data will not help much.
- **Validation still rising at 100%** -> more data (or more representative data) is likely to help.
- **Large persistent gap at 100%** -> variance; the fix is regularisation/simpler model, not more data.
- **Both curves low and flat** -> bias; the fix is better features or a richer model.

The table prints the gap at each training size and a verdict for the final point.

In [9]:
learning_tables = {}
learning_verdicts = {}

for model_name in cv_summary.head(2)["Model"]:
    curve_jobs = 1 if model_name == "SVM (RBF)" else -1

    train_sizes, train_scores, val_scores = learning_curve(
        estimator=searches[model_name].best_estimator_,
        X=X_train,
        y=y_train,
        train_sizes=np.linspace(0.1, 1.0, 6),
        cv=cv,
        scoring="roc_auc",
        n_jobs=curve_jobs,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    table = pd.DataFrame({
        "Training rows": train_sizes.astype(int),
        "Train ROC-AUC": train_scores.mean(axis=1),
        "Validation ROC-AUC": val_scores.mean(axis=1),
    })
    table["Gap"] = table["Train ROC-AUC"] - table["Validation ROC-AUC"]

    last_gap = table["Gap"].iloc[-1]
    last_gain = (
        table["Validation ROC-AUC"].iloc[-1]
        - table["Validation ROC-AUC"].iloc[-2]
    )
    last_val = table["Validation ROC-AUC"].iloc[-1]

    if last_gap > 0.05:
        verdict = "Variance gap persists -> more regularisation or simpler model"
    elif last_gain > 0.005:
        verdict = "Validation still rising -> more data is likely to help"
    elif last_val < 0.70:
        verdict = "Both scores low -> bias; needs better features or a richer model"
    else:
        verdict = "Curves converged -> diminishing returns from more data"

    learning_tables[model_name] = table
    learning_verdicts[model_name] = verdict

    print("=" * 72)
    print(model_name)
    print(table.round(4).to_string(index=False))
    print(f"Last validation gain: {last_gain:+.4f}")
    print("Verdict:", verdict)
    print()

Random Forest
 Training rows  Train ROC-AUC  Validation ROC-AUC    Gap
           450         0.9434              0.8339 0.1096
          1261         0.9328              0.8390 0.0939
          2073         0.9317              0.8426 0.0891
          2884         0.9281              0.8450 0.0831
          3695         0.9237              0.8459 0.0778
          4507         0.9207              0.8461 0.0746
Last validation gain: +0.0002
Verdict: Variance gap persists -> more regularisation or simpler model



Logistic Regression
 Training rows  Train ROC-AUC  Validation ROC-AUC    Gap
           450         0.8626              0.8263 0.0363
          1261         0.8454              0.8401 0.0053
          2073         0.8493              0.8410 0.0083
          2884         0.8496              0.8446 0.0050
          3695         0.8484              0.8453 0.0031
          4507         0.8498              0.8462 0.0036
Last validation gain: +0.0009
Verdict: Curves converged -> diminishing returns from more data



## 10. Lock the champion and evaluate it once on the holdout

The best configuration is selected by CV ROC-AUC (recall breaks ties). It has already been refit on the full training set by `GridSearchCV(refit=True)`, so the test set is used exactly once here.

Because the SVM is tuned without probability outputs, it is wrapped in `CalibratedClassifierCV` (sigmoid, 5-fold on the training split) **only if it wins**. That gives the saved pipeline a real `predict_proba` without paying calibration cost for every grid candidate.

Why ROC-AUC for selection: it measures ranking quality across all thresholds, which matters when the churn threshold is tuned later for the business. Recall/precision/F1 are still reported to show the operational trade-off at the default 0.5 threshold.

In [10]:
champion_name = cv_summary.loc[0, "Model"]
champion = searches[champion_name].best_estimator_

calibration_note = "Not needed - champion natively supports predict_proba"

if hasattr(champion, "predict_proba"):
    champion_model = champion
else:
    # SVC was tuned without probability; calibrate it once on the training split.
    champion_model = CalibratedClassifierCV(champion, method="sigmoid", cv=5)
    champion_model.fit(X_train, y_train)
    calibration_note = "Sigmoid calibration (cv=5) applied to SVC on the training split"

y_pred = champion_model.predict(X_test)
y_prob = champion_model.predict_proba(X_test)[:, 1]

test_metrics = {
    "Model": champion_name,
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred, zero_division=0),
    "Recall": recall_score(y_test, y_pred, zero_division=0),
    "F1": f1_score(y_test, y_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, y_prob),
}

pd.DataFrame([test_metrics]).round(4)

,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,Random Forest,0.7502,0.5198,0.7727,0.6215,0.8419


In [11]:
print("Champion:", champion_name)
print("Best parameters:", searches[champion_name].best_params_)
print()
print(classification_report(y_test, y_pred, target_names=["No Churn", "Churn"]))
print("Confusion matrix [[TN, FP], [FN, TP]]:")
print(confusion_matrix(y_test, y_pred))

Champion: Random Forest
Best parameters: {'classifier__max_depth': 10, 'classifier__min_samples_leaf': 5, 'classifier__n_estimators': 200}

              precision    recall  f1-score   support

    No Churn       0.90      0.74      0.81      1035
       Churn       0.52      0.77      0.62       374

    accuracy                           0.75      1409
   macro avg       0.71      0.76      0.72      1409
weighted avg       0.80      0.75      0.76      1409

Confusion matrix [[TN, FP], [FN, TP]]:
[[768 267]
 [ 85 289]]


## 11. Version and document the champion

The tuned pipeline is saved as `models/churn_v3.joblib`, leaving Day 1/Day 2 artifacts untouched. A model card is generated alongside it with the exact hyperparameters, CV leaderboard, holdout metrics and bias-variance verdict — paste it into the README so the decision is reproducible.

In [12]:
MODELS_DIR_CANDIDATES = [Path("../models"), Path("models")]
MODELS_DIR = next(
    (path for path in MODELS_DIR_CANDIDATES if path.exists()),
    Path("../models"),
)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

CHAMPION_PATH = MODELS_DIR / "churn_v3.joblib"
CV_RESULTS_PATH = MODELS_DIR / "day3_cv_results.csv"
TEST_METRICS_PATH = MODELS_DIR / "day3_test_metrics.csv"
BEST_PARAMS_PATH = MODELS_DIR / "day3_best_params.json"
MODEL_CARD_PATH = MODELS_DIR / "day3_model_card.md"

# 1. Champion pipeline (calibrated if the SVM won)
joblib.dump(champion_model, CHAMPION_PATH)

# 2. CV leaderboard (best config per model)
cv_summary.drop(columns=["Best Parameters"]).to_csv(CV_RESULTS_PATH, index=False)

# 3. One-shot holdout metrics
pd.DataFrame([test_metrics]).to_csv(TEST_METRICS_PATH, index=False)

# 4. Best hyperparameters, JSON-safe
best_params_payload = {
    "champion": champion_name,
    "selection_metric": "roc_auc",
    "cv_folds": cv.get_n_splits(),
    "random_state": RANDOM_STATE,
    "selection": {
        key: float(value)
        for key, value in cv_summary.iloc[0][
            ["CV ROC-AUC", "CV ROC-AUC Std", "CV Recall", "CV F1", "Train-CV Gap"]
        ].items()
    }
    | {"diagnosis": cv_summary.iloc[0]["Diagnosis"]},
    "best_params": {
        name: {
            key: (value.item() if hasattr(value, "item") else value)
            for key, value in search.best_params_.items()
        }
        for name, search in searches.items()
    },
}
BEST_PARAMS_PATH.write_text(
    json.dumps(best_params_payload, indent=2, default=str),
    encoding="utf-8",
)


def markdown_table(frame):
    columns = list(frame.columns)
    lines = [
        "| " + " | ".join(columns) + " |",
        "|" + "|".join(["---"] * len(columns)) + "|",
    ]
    for _, row in frame.iterrows():
        lines.append("| " + " | ".join(str(row[column]) for column in columns) + " |")
    return "\n".join(lines)


card_table = cv_summary[[
    "Model", "Best Parameters", "CV ROC-AUC", "CV ROC-AUC Std",
    "CV Recall", "CV F1", "Train ROC-AUC", "Train-CV Gap", "Diagnosis",
]].round(4)

holdout_lines = "\n".join(
    f"- **{key}:** {value:.4f}" if isinstance(value, float) else f"- **{key}:** {value}"
    for key, value in test_metrics.items()
)

curve_lines = "\n".join(
    f"- **{name}:** {verdict}" for name, verdict in learning_verdicts.items()
)

model_card = f'''# Day 3 model card - churn_v3

## Champion

- **Model:** {champion_name}
- **Best hyperparameters:** {searches[champion_name].best_params_}
- **Probability calibration:** {calibration_note}
- **Saved artifact:** `models/churn_v3.joblib`
- **Dataset:** Telco Customer Churn, {len(df)} clean rows, {X.shape[1]} raw features
- **CV protocol:** StratifiedKFold(n_splits=5, shuffle=True, random_state={RANDOM_STATE})
- **Selection metric:** ROC-AUC (recall as tie-breaker)

## Cross-validation leaderboard

{markdown_table(card_table)}

## Holdout test set (20% stratified, evaluated once)

{holdout_lines}

## Bias-variance diagnosis

{cv_summary[["Model", "Diagnosis"]].to_string(index=False)}

## Learning-curve verdicts

{curve_lines}

## Reproduce

Run `notebooks/retailmind_day3_tuning.ipynb` from the repo root or from `notebooks/`.
The test split is fixed with `random_state={RANDOM_STATE}`, so the holdout numbers are reproducible.
'''

MODEL_CARD_PATH.write_text(model_card, encoding="utf-8")

print("Saved:")
for path in [CHAMPION_PATH, CV_RESULTS_PATH, TEST_METRICS_PATH, BEST_PARAMS_PATH, MODEL_CARD_PATH]:
    print(" -", path.resolve())

Saved:
 - C:\Users\ANSHU SHAW\OneDrive\Desktop\RETAILMIND_AI\models\churn_v3.joblib
 - C:\Users\ANSHU SHAW\OneDrive\Desktop\RETAILMIND_AI\models\day3_cv_results.csv
 - C:\Users\ANSHU SHAW\OneDrive\Desktop\RETAILMIND_AI\models\day3_test_metrics.csv
 - C:\Users\ANSHU SHAW\OneDrive\Desktop\RETAILMIND_AI\models\day3_best_params.json
 - C:\Users\ANSHU SHAW\OneDrive\Desktop\RETAILMIND_AI\models\day3_model_card.md


## 12. Verify the saved artifact round-trips

A model is only shipped if it can be reloaded and used. This reloads the joblib file and checks that its predictions match the in-memory champion for a few customers.

In [13]:
reloaded = joblib.load(CHAMPION_PATH)

sample = X_test.head(5)
in_memory = champion_model.predict_proba(sample)[:, 1]
from_disk = reloaded.predict_proba(sample)[:, 1]

print("Predictions match after reload:", np.allclose(in_memory, from_disk))
pd.DataFrame({
    "customer_index": sample.index,
    "churn_probability": from_disk.round(4),
    "prediction": reloaded.predict(sample),
})

Predictions match after reload: True


,customer_index,churn_probability,prediction
0,437,0.0302,0
1,2280,0.8457,1
2,2235,0.1404,0
3,4460,0.5486,1
4,3761,0.0557,0


## 13. Day 3 summary

The cell below prints the decisions that belong in the README: champion, hyperparameters, CV and holdout scores, the bias-variance read, and what the learning curve says to do next.

In [14]:
top_row = cv_summary.iloc[0]

print("DAY 3 - CHURN MODEL DECISION")
print("=" * 60)
print(f"Champion            : {top_row['Model']}")
print(f"Hyperparameters     : {top_row['Best Parameters']}")
print(f"CV ROC-AUC          : {top_row['CV ROC-AUC']:.4f} (+/- {top_row['CV ROC-AUC Std']:.4f})")
print(f"CV Recall / F1      : {top_row['CV Recall']:.4f} / {top_row['CV F1']:.4f}")
print(f"Train-CV gap        : {top_row['Train-CV Gap']:+.4f}")
print(f"Diagnosis           : {top_row['Diagnosis']}")
print(f"Holdout ROC-AUC     : {test_metrics['ROC-AUC']:.4f}")
print(f"Holdout Recall      : {test_metrics['Recall']:.4f}")
print(f"Artifact            : {CHAMPION_PATH}")
print()
print("Next actions from learning curves:")
for name, verdict in learning_verdicts.items():
    print(f" - {name}: {verdict}")
print()
print("Day 4 direction: feature engineering, PCA, and boosting/bagging ensembles.")

DAY 3 - CHURN MODEL DECISION
Champion            : Random Forest
Hyperparameters     : {'classifier__max_depth': 10, 'classifier__min_samples_leaf': 5, 'classifier__n_estimators': 200}
CV ROC-AUC          : 0.8464 (+/- 0.0093)
CV Recall / F1      : 0.7746 / 0.6314
Train-CV gap        : +0.0738
Diagnosis           : High variance (overfitting)
Holdout ROC-AUC     : 0.8419
Holdout Recall      : 0.7727
Artifact            : ..\models\churn_v3.joblib

Next actions from learning curves:
 - Random Forest: Variance gap persists -> more regularisation or simpler model
 - Logistic Regression: Curves converged -> diminishing returns from more data

Day 4 direction: feature engineering, PCA, and boosting/bagging ensembles.
